# Training code 

## Import

In [1]:
import aegnn
import wandb
import datetime
import lightning.pytorch as pl
import lightning.pytorch.loggers
import torch

from pathlib import Path

import os

In [2]:
os.environ["AEGNN_DATA_DIR"] = "/home/kevin-imagine/Documents/event_graph/data"
print(os.environ["AEGNN_DATA_DIR"])

/home/kevin-imagine/Documents/event_graph/data


## Classification

### Parameters

In [3]:
params = {"model":"graph_res",
          "task":"",
          "dim":3,
          "seed":12345,
          "lr": 1e-3,
          "weight_decay":5e-3,
          "eta_min":0.0,
          "max_epochs":100,
          "Trainer":{
            "max_epochs":100,
            "overfit_batches":0,
            "log_every_n_steps":10,
            "gradient_clip_val":0,
            "limit_train_batches":1,
            "limit_val_batches":1
          },
          "log-gradients":True,
          "profile":True,
          "debug":True,
          "gpu": True if torch.cuda.is_available() else False,
          "dataset":"ncars",
          "Data":{
              "batch_size":8,
              "num_workers":8,
              "pin_memory":False
          }          
}

### Model and dataset

In [4]:
data_module = aegnn.datasets.NCars(shuffle = True,
                                   **params["Data"],
                                   )

In [5]:
data_module.setup()

In [6]:
model = aegnn.models.RecognitionModel(params["model"],
                                      params["dataset"],
                                      num_classes=data_module.num_classes,
                                      img_shape=data_module.dims,
                                      dim = params["dim"],
                                      bias = True,
                                      root_weight = True,                                     
                                      )


### Loggers

In [7]:
log_settings = wandb.Settings(start_method='thread')
log_dir = "../training_log_aegnn/"
loggers = [aegnn.utils.loggers.LoggingLogger(None if params["debug"] else log_dir, name="debug")]

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


In [8]:
project = f"aegnn-{params['dataset']}-{params['task']}"
experiment_name = datetime.datetime.now().strftime("%Y%m%d%H%M%S")


In [9]:
if not params["debug"]:
    wandb_logger = pl.loggers.WandbLogger(project=project, save_dir=log_dir, settings=log_settings)
    if params["log-gradients"]:
        wandb_logger.watch(model, log="gradients")
    loggers.append(wandb_logger)
# loggers = pl.loggers.LoggerCollection(loggers)

In [10]:
checkpoint_path = os.path.join(log_dir, "checkpoints", params["dataset"], params["task"], experiment_name)
Path(checkpoint_path).mkdir(parents=True,exist_ok=True)

In [11]:
callbacks = [
        pl.callbacks.LearningRateMonitor(),
        aegnn.utils.callbacks.BBoxLogger(classes=data_module.classes),
        # aegnn.utils.callbacks.PHyperLogger(args),
        aegnn.utils.callbacks.EpochLogger(),
        aegnn.utils.callbacks.FileLogger([model, model.model, data_module]),
        aegnn.utils.callbacks.FullModelCheckpoint(dirpath=checkpoint_path)
    ]

In [12]:
trainer_kwargs = {
    "accelerator":"gpu" if params["gpu"] else "cpu",
    "profiler": "simple" if params["profile"] else False,
}

In [13]:
trainer = pl.Trainer(logger=loggers,
                     callbacks=callbacks,
                     **params["Trainer"],
                     **trainer_kwargs
                     )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.
`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


In [14]:
trainer.fit(model, datamodule=data_module)

You are using a CUDA device ('NVIDIA RTX PRO 1000 Blackwell Generation Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/training/*
RAW FILer length 12961


100%|██████████| 12961/12961 [00:00<00:00, 284108.07it/s]

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/validation/*
RAW FILer length 2462



100%|██████████| 2462/2462 [00:00<00:00, 129228.32it/s]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | criterion | CrossEntropyLoss | 0      | train | 0    
1 | model     | GraphRes         | 30.4 K | train | 0    
---------------------------------------------------------------
30.4 K    Trainable params
0         Non-trainable params
30.4 K    Total params
0.121     Total estimated model params size (MB)
40        Modules in train mode
0         Modules in eval mode
0         Total Flops


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
FIT Profiler Report

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Action                                                                                                                                                             	|  Mean duration (s)	|  Num calls      	|  Total time (s) 	|  Percentage %   	|
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|  Total                                                                                                                                             

In [15]:
import torch

In [16]:
torch.cuda.is_available()

True

In [17]:
torch.cuda.get_device_name(0)

'NVIDIA RTX PRO 1000 Blackwell Generation Laptop GPU'

In [18]:
import lightning.pytorch as pl